In [ ]:
from pathlib import Path
import sys
import json

#will run in any directory
cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd if (cwd / "rag" / "offer_gen.py").exists() else cwd.parent

print("Project root:", PROJECT_ROOT)
print("Root contents:", list(PROJECT_ROOT.iterdir())[:20])

if not (PROJECT_ROOT / "rag" / "offer_gen.py").exists():
    raise RuntimeError(
        f"Cannot find rag/offer_gen.py in {PROJECT_ROOT}. "
        "Check that the project files are present."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from rag.offer_gen import (
    load_policy_documents,
    split_policy_documents,
    build_vector_store,
    retrieve_policy_context,
    generate_offer,
    rag_invoke_fn,
)

print("Policy directory:", PROJECT_ROOT / "northstar-policy-docs")


Project root: c:\Users\sarah\cs401r--shc49-
Root contents: [WindowsPath('c:/Users/sarah/cs401r--shc49-/.dist'), WindowsPath('c:/Users/sarah/cs401r--shc49-/.git'), WindowsPath('c:/Users/sarah/cs401r--shc49-/.gitignore'), WindowsPath('c:/Users/sarah/cs401r--shc49-/.localstack'), WindowsPath('c:/Users/sarah/cs401r--shc49-/.venv'), WindowsPath('c:/Users/sarah/cs401r--shc49-/.vscode'), WindowsPath('c:/Users/sarah/cs401r--shc49-/aws-account-setup.md'), WindowsPath('c:/Users/sarah/cs401r--shc49-/churn_training_skeleton.py'), WindowsPath('c:/Users/sarah/cs401r--shc49-/docker-compose.yml'), WindowsPath('c:/Users/sarah/cs401r--shc49-/docs'), WindowsPath('c:/Users/sarah/cs401r--shc49-/docs\uf00d'), WindowsPath('c:/Users/sarah/cs401r--shc49-/evaluation_harness.py'), WindowsPath('c:/Users/sarah/cs401r--shc49-/evaluation_results'), WindowsPath('c:/Users/sarah/cs401r--shc49-/glue-scripts'), WindowsPath('c:/Users/sarah/cs401r--shc49-/infrast'), WindowsPath('c:/Users/sarah/cs401r--shc49-/infrastructure

In [2]:
documents = load_policy_documents()
chunks = split_policy_documents(documents)

print(f"Policy documents loaded: {len(documents)}")
print(f"Policy chunks created: {len(chunks)}")

for document in documents:
    print(f"- {document.metadata['source']}: "
          f"{len(document.page_content)} characters")

Policy documents loaded: 3
Policy chunks created: 27
- loyalty-program-terms.md: 5959 characters
- shipping-policy.md: 4568 characters
- return-policy.md: 6184 characters


In [3]:
customer = {
    "customer_id": "CUST-10000317",
    "loyalty_tier": "Gold",
    "churn_probability": 0.81,
    "days_since_last_purchase": 47,
    "total_lifetime_value": 3120.40,
    "avg_order_value": 186.25,
    "category_diversity_score": 0.5,
    "online_to_store_ratio": 0.72,
    "top_categories": ["Outdoor", "Footwear"],
}

question = (
    "Generate a personalized retention offer for this at-risk "
    "Gold member who shops mostly online for outdoor gear and footwear."
)

# Embedding creation calls Amazon Bedrock.
vector_store = build_vector_store(chunks)

retrieved = retrieve_policy_context(
    vector_store, question, customer
)

print(f"Retrieved {len(retrieved)} policy chunks")

for i, document in enumerate(retrieved, start=1):
    print(f"\n--- Result {i}: {document.metadata.get('source')} ---")
    print(document.page_content[:1000])

Retrieved 5 policy chunks

--- Result 1: loyalty-program-terms.md ---
Offer codes are void if the account is closed, if the offer's stated conditions are not met, or after the stated expiry date. Expired offers **cannot be reinstated by customer service or by any automated system.**

### 5.1 Retention Offers

Retention offers are generated for members identified as at risk of lapsing. These offers:

- Are valid for **45 days** from issue unless stated otherwise
- Are limited to **one active retention offer per member** at a time
- May not be combined with another retention offer
- Do not extend, alter, or waive any provision of the returns policy

--- Result 2: loyalty-program-terms.md ---
Benefits are tied to the member's tier at the time the benefit is used, except the return window, which is fixed at the tier held **at the time of purchase** (see POL-RET-004 Â§1.1).

### 4.1 Annual Gear Credit

Platinum members receive a **$100 gear credit** each January 1. The credit expires Decemb

In [4]:
result = generate_offer(customer, question, vector_store)

print("QUESTION:")
print(question)

print("\nGENERATED OFFER:")
print(result["answer"])

print("\nRETRIEVED CONTEXTS:")
for i, context in enumerate(result["contexts"], start=1):
    print(f"\nContext {i}:\n{context[:800]}")

QUESTION:
Generate a personalized retention offer for this at-risk Gold member who shops mostly online for outdoor gear and footwear.

GENERATED OFFER:
Subject: Your Gold Benefits Are Waitingâ€”Free Return Shipping on Your Next Order

Body: We'd love to see you back in Outdoor and Footwear gear. As a Gold member, you enjoy free return shipping on eligible purchases [POL-LOY-011], so you can shop with confidence knowing returns are hassle-free. Come explore what's new, and let us know if you have questionsâ€”we're here to help.

Rationale: The customer is labeled Gold with 47 days since last purchase and concentrated shopping in Outdoor and Footwear (diversity score 0.50). Free return shipping is a verified Gold benefit applicable now, aligns with the customer's online-heavy ratio (0.72), and addresses a common friction point for online shoppers without requiring a discount or invented promotion. The benefit is policy-backed and relevant to the customer's demonstrated category preferenc

In [5]:
bronze_customer = {
    "customer_id": "CUST-10000904",
    "loyalty_tier": "Bronze",
    "churn_probability": 0.74,
    "days_since_last_purchase": 68,
    "total_lifetime_value": 312.80,
    "avg_order_value": 41.15,
    "category_diversity_score": 0.125,
    "online_to_store_ratio": 0.20,
    "top_categories": ["Apparel"],
}

bronze_question = (
    "Generate a retention offer for this Bronze member who shops in store, "
    "buys only apparel, and has not purchased in over two months."
)

bronze_result = generate_offer(
    bronze_customer,
    bronze_question,
    vector_store,
)

print("QUESTION:")
print(bronze_question)
print("\nGENERATED OFFER:")
print(bronze_result["answer"])
print("\nCONTEXTS RETURNED:", len(bronze_result["contexts"]))

QUESTION:
Generate a retention offer for this Bronze member who shops in store, buys only apparel, and has not purchased in over two months.

GENERATED OFFER:
Subject: We miss youâ€”come back to Apparel and earn rewards

Body: It's been a while since we've seen you, and we'd love to welcome you back to NorthStar. As a Bronze member, you earn 1 point per $1 on all qualifying apparel purchases, and when your order reaches $75 or more, you'll receive free standard shipping [POL-LOY-011]. Shop apparel today and start earning toward your next reward.

Rationale: The customer is Bronze-tier (unverified), has not purchased in 68 days, and concentrates purchases in Apparel (low category diversity). Bronze members qualify for conditional free standard shipping at $75+ and a 1Ã— points earn rate. The $75 threshold is relevant given the customer's $41.15 average order value, making it an achievable incentive to motivate return. No retention offer code is issued because the customer profile does n